# Tutorial 01: Spatial Analysis of Visium H&E Data

---


**Goal:** Explore the relationship between tissue morphology and gene expression using Squidpy.

**Dataset:** Coronal section of the Mouse Brain (10x Genomics).

**Key Methods:** Image feature extraction, neighborhood enrichment, and Moran's I spatial autocorrelation.


---
**Javeria Butt**


### **1. Imports**

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import warnings
warnings.filterwarnings('ignore')

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

### **2. Load Data**

The dataset is a pre-processed Visium slide of a mouse brain coronal section with pre-annotated clusters. It is downloaded automatically by `sq.datasets`.

In [ ]:
img   = sq.datasets.visium_hne_image()
adata = sq.datasets.visium_hne_adata()

print(adata)
print(f"\nClusters: {adata.obs['cluster'].nunique()}")
print(adata.obs['cluster'].value_counts())

### **3. Spatial Visualization of Cluster Annotations**

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color='cluster',
    save='../results/figures/01_visium_hne/spatial_clusters.png'
)

### **4. Image Features**
Squidpy can extract image features from the high-resolution tissue image for each Visium spot. This allows us to incorporate tissue morphology information alongside gene expression.

We extract summary features (mean, std, quantiles of pixel intensity) at two spatial scales — the features at larger scales capture broader tissue context.

In [ ]:
# Extract summary features at two scales
for scale in [1.0, 2.0]:
    feature_name = f"features_summary_scale{scale}"
    sq.im.calculate_image_features(
        adata,
        img.compute(),
        features='summary',
        key_added=feature_name,
        n_jobs=4,
        scale=scale,
    )

# Combine into one feature matrix
adata.obsm['features'] = pd.concat(
    [adata.obsm[f] for f in adata.obsm.keys() if 'features_summary' in f],
    axis='columns',
)
adata.obsm['features'].columns = ad.utils.make_index_unique(
    adata.obsm['features'].columns
)

print(f"Image features shape: {adata.obsm['features'].shape}")

### **5. Cluster Spots by Image Features**

In [ ]:
def cluster_features(features: pd.DataFrame, like=None) -> pd.Series:
    """Leiden clustering of image features."""
    if like is not None:
        features = features.filter(like=like)
    tmp_adata = ad.AnnData(features)
    sc.pp.scale(tmp_adata)
    sc.pp.pca(tmp_adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(tmp_adata)
    sc.tl.leiden(tmp_adata, key_added='leiden_features')
    return tmp_adata.obs['leiden_features']

adata.obs['image_cluster'] = cluster_features(adata.obsm['features'])

print(f"Image-based clusters: {adata.obs['image_cluster'].nunique()}")

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color=['cluster', 'image_cluster'],
    save='../results/figures/01_visium_hne/cluster_comparison.png'
)


### **6. Spatial Graph Analysis**

### 6a. Build Spatial Neighbor Graph

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type='grid', n_neighs=6)

### 6b. Neighborhood Enrichment

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key='cluster')
sq.pl.nhood_enrichment(
    adata,
    cluster_key='cluster',
    method='average',
    save='../results/figures/01_visium_hne/nhood_enrichment.png'
)

### 6c. Co-occurrence Score

In [ ]:

sq.gr.co_occurrence(adata, cluster_key='cluster')
sq.pl.co_occurrence(
    adata,
    cluster_key='cluster',
    clusters='Hippocampus',
    save='../results/figures/01_visium_hne/co_occurrence.png'
)

### 6d. Interaction Matrix

In [ ]:
sq.gr.interaction_matrix(adata, cluster_key='cluster')
sq.pl.interaction_matrix(
    adata,
    cluster_key='cluster',
    save='../results/figures/01_visium_hne/interaction_matrix.png'
)


### 6e. Centrality Scores

In [ ]:

sq.gr.centrality_scores(adata, cluster_key='cluster')
sq.pl.centrality_scores(
    adata,
    cluster_key='cluster',
    save='../results/figures/01_visium_hne/centrality_scores.png'
)

### 6f. Ripley's Statistics

In [ ]:
sq.gr.ripley(adata, cluster_key='cluster', mode='L')
sq.pl.ripley(
    adata,
    cluster_key='cluster',
    mode='L',
    save='../results/figures/01_visium_hne/ripley_L.png'
)

### 6g. Spatial Autocorrelation — Moran's I

In [ ]:
sq.gr.spatial_autocorr(adata, mode='moran', n_perms=100, n_jobs=4)

print("Top 10 spatially variable genes (Moran's I):")
print(adata.uns['moranI'].head(10))


In [ ]:

# Visualize top spatially variable genes on tissue
top_svgs = adata.uns['moranI'].head(6).index.tolist()
sq.pl.spatial_scatter(
    adata,
    color=top_svgs,
    save='../results/figures/01_visium_hne/top_spatial_genes.png'
)

### **7. Save Results**

In [ ]:
# Save to the results folder sitting in /content
adata.write('results/visium_hne_analyzed.h5ad')
print('Saved successfully to /content/results/')